# MathPaper AI — Colab Runner

Run the agentic RAG system on Google's hardware, so it works regardless of your
own laptop's specs. Three ways to run, pick whichever you like:

1. **Zero setup** — retrieval benchmark + orchestration tests, no API key, no GPU.
2. **Free cloud API** — full-quality answers via Groq (no credit card).
3. **Local models on Colab's free T4 GPU** — run Qwen/Llama locally via Ollama.

**Tip:** for options 2 & 3 that need the GPU, set the runtime first:
`Runtime → Change runtime type → T4 GPU`.


## 0. Get the code

Clone the repo. If you've pushed your own fork, change the URL.

In [ ]:
# If you pushed the repo to GitHub, put your URL here:
REPO_URL = "https://github.com/YOUR_USERNAME/mathpaper-ai.git"

import os
if not os.path.exists("mathpaper-ai"):
    # Try to clone; if the repo isn't public yet, upload the zip instead (see next cell).
    !git clone $REPO_URL 2>/dev/null && echo "Cloned." || echo "Clone failed — use the upload cell below."
%cd mathpaper-ai 2>/dev/null || echo "Not cloned yet."


### (Alternative) Upload the zip instead of cloning
If your repo isn't on GitHub yet, run this cell, upload `mathpaper-ai.zip`, and it
will unpack it.

In [ ]:
# Uncomment to upload the zip instead of cloning:
# from google.colab import files
# up = files.upload()               # choose mathpaper-ai.zip
# import zipfile, io
# name = list(up.keys())[0]
# with zipfile.ZipFile(io.BytesIO(up[name])) as z: z.extractall(".")
# %cd mathpaper-ai-repo


## 1. Install dependencies

In [ ]:
!pip install -q -e ".[dev]"
print("Installed.")

## 2. Zero-setup checks (no API key, no GPU)
Runs the orchestration tests (mocked LLM) and the retrieval benchmark. This proves
the pipeline logic works and regenerates the result charts.

In [ ]:
!pytest tests/ -v

In [ ]:
!python evaluate.py
from IPython.display import Image, display
display(Image("results/retrieval_comparison.png"))
display(Image("results/latency_orchestration.png"))

---
## 3. Full pipeline with a FREE cloud API (recommended)

Get a free Groq key at **console.groq.com** (no credit card), paste it below, and
run the whole multi-agent pipeline with real, full-quality answers. This uses **no
GPU**, so it works on any Colab runtime.

In [ ]:
import os, getpass
os.environ["LLM_PROVIDER"] = "groq"
os.environ["GROQ_API_KEY"] = getpass.getpass("Paste your Groq API key: ")

# smoke test the adapter
!python -m mathpaper.llm

In [ ]:
# Run the full agent pipeline on a sample question
!python -m mathpaper.agents

### Ask your own question
Edit the question and run. Watch the agent trace (which agents fired) then the answer.

In [ ]:
from mathpaper import PlanningAgent, HybridRetriever, load_demo_corpus

planner = PlanningAgent(HybridRetriever(load_demo_corpus()))

question = "Why is cross entropy used instead of mean squared error?"   # <-- edit me
state = planner.run(question)

print("AGENT TRACE")
print("-" * 50)
for step in state.trace:
    print(" ", step)
print("\nANSWER")
print("-" * 50)
print(state.answer)

---
## 4. (Optional) Run LOCAL models on Colab's free T4 GPU

This installs Ollama inside Colab and runs open models on the T4 GPU — no API key,
fully offline once downloaded. **Requires the GPU runtime**
(`Runtime → Change runtime type → T4 GPU`).

The T4 has ~16GB VRAM, enough for `qwen2.5:7b` (strong agents) plus a small model.

In [ ]:
# Confirm a GPU is attached
!nvidia-smi -L || print("No GPU! Set Runtime -> Change runtime type -> T4 GPU, then rerun.")

In [ ]:
# Install Ollama and start its server in the background
!curl -fsSL https://ollama.com/install.sh | sh
import subprocess, time
subprocess.Popen(["ollama", "serve"])
time.sleep(5)
print("Ollama server started.")

In [ ]:
# Pull the models (first run downloads a few GB; cached for the session)
!ollama pull qwen2.5:7b        # strong: planner, verifier, generator
!ollama pull llama3.2:3b       # small: classifier, memory
print("Models ready.")

In [ ]:
# Point the adapter at Ollama and run the pipeline locally on the T4
import os
os.environ["LLM_PROVIDER"] = "ollama"
os.environ.pop("GROQ_API_KEY", None)

# The llm.py Ollama config defaults to qwen2.5:3b/7b names; qwen2.5:7b + llama3.2:3b
# are close matches. If you pulled different tags, edit PROVIDERS['ollama'] in
# src/mathpaper/llm.py accordingly.
!python -m mathpaper.agents

---
## Notes & gotchas
- **Session limits:** free Colab disconnects after ~90 min idle / ~12 h max, and the
  disk wipes on disconnect. Re-run the setup cells each new session, or mount Google
  Drive to cache Ollama models (`~/.ollama`).
- **GPU quota:** the free T4 has a daily cap. If you hit it, use the Groq path (§3) —
  it needs no GPU.
- **Which to use?** For developing and grading answers, the Groq path (§3) is fastest
  and highest quality. The Ollama path (§4) is for the "I ran it fully local/offline"
  demonstration.
- **Privacy:** free API tiers may train on your prompts — don't paste anything private.
